---
# <div align="center"><font color='red'>  </font></div>
# <div align="center"><font color='red'> COSC 2779/2972 | Deep Learning  </font></div>
## <div align="center"> <font color='red'> Week 7 Lecture Example: **Sequential Data & Recurrent Networks**</font></div>
---

This week we model **sequences**; data where *order* matters and the length varies. We build the core ideas from scratch so the mechanics are visible:

1. **Representing tokens**: one-hot vs learned **embeddings**.
2. **The recurrent neural network (RNN)**: a hidden state carried through time, built cell-by-cell.
3. The **long-range problem** (vanishing gradients) and how **GRU** and **LSTM** gates fix it.
4. Two output patterns: **sequence classification** (one label per sequence, *many-to-one*) and **token classification** (one label per position, *many-to-many*).

The goal is not accuracy on a hard dataset — it is to *see* how these models work. So we use tiny **synthetic tasks** on which we can watch memory succeed and fail.

# Setup Notebook

In [1]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
torch.manual_seed(42); np.random.seed(42)

In [2]:
from itertools import cycle

def plotter(history_hold, metric='loss', ylim=[0.0, 1.0]):
    cycol = cycle('bgrcmk')
    for name, hist in history_hold.items():
        y_train = hist[metric]; y_val = hist['val_' + metric]
        x = np.arange(len(y_val)); c = next(cycol)
        plt.plot(x, y_train, c + '-',  label=name + '_train')
        plt.plot(x, y_val,   c + '--', label=name + '_val')
    plt.xlabel('epoch'); plt.ylabel(metric); plt.ylim(ylim); plt.legend(); plt.show()

def n_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Representing tokens: one-hot vs. embeddings

A network needs numbers, not symbols. The simplest encoding is **one-hot**: token *i* becomes a vector that is 1 in slot *i* and 0 everywhere else. It works, but it is huge (length = vocabulary size), sparse, and crucially **every pair of tokens is equally dissimilar** (all dot products are 0). It carries no notion that two words might be related.

An **embedding** instead maps each token id to a short, *dense, learned* vector. `nn.Embedding` is literally a lookup table: row *i* is the vector for token *i*, and the rows are trained by gradient descent.

In [3]:
vocab, d_model = 6, 4
onehot = F.one_hot(torch.arange(vocab)).float()
print('one-hot for token 3:', onehot[3].tolist(), ' (length =', vocab, ')')

emb = nn.Embedding(vocab, d_model)
print('embedding for token 3:', np.round(emb(torch.tensor(3)).detach().numpy(), 2))
# an embedding lookup is exactly: (one-hot row) x (embedding weight matrix)
print('lookup == one-hot @ weight :',
      torch.allclose(emb(torch.tensor(3)), onehot[3] @ emb.weight))

# Part 1 — Sequence classification: remembering across time

**Task (remember the order).** Every sequence is a run of random "noise" tokens (`2..11`), except the **first two positions**, which always hold the two marker tokens `0` and `1` — in one order or the other. The **label is which marker came first**.

Because *both* markers are always present, a model that only looks at *which* tokens occur (a bag of tokens) learns nothing: the multiset is identical either way. Only a model that respects **order** can tell the two cases apart and, as we will see, even an ordered RNN forgets that one bit once the sequence gets long.

In [4]:
V = 12          # tokens 0,1 = order markers (always at positions 0,1); 2..11 = noise
def make_recall(n, L):
    X = np.random.randint(2, V, size=(n, L)).astype(np.int64)   # noise everywhere
    y = np.random.randint(0, 2, size=n).astype(np.int64)         # the label
    X[:, 0] = np.where(y == 1, 0, 1)    # the two markers lead the sequence, in an
    X[:, 1] = np.where(y == 1, 1, 0)    # order that encodes the label (0,1)->1  (1,0)->0
    return X, y

Xr, yr = make_recall(4, 12)
for i in range(4):
    print('label =', yr[i], '| sequence:', Xr[i], '  (both markers 0 and 1 always lead)')

## Data & training helpers
Standard `DataLoader`s and a `train_and_track` helper returning a `history` dict (loss/accuracy/val). RNNs can have exploding gradients, so we clip the gradient norm — a standard, useful default.

In [5]:
def make_loaders(X, y, batch_size=128, shuffle=True):
    return DataLoader(TensorDataset(torch.from_numpy(X), torch.from_numpy(y)),
                      batch_size=batch_size, shuffle=shuffle)

def _epoch(model, loader, criterion, optimizer=None):
    train = optimizer is not None
    model.train() if train else model.eval()
    tot_loss = tot_correct = tot_n = 0
    torch.set_grad_enabled(train)
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb); loss = criterion(logits, yb)
        if train:
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # clip exploding grads
            optimizer.step()
        tot_loss += loss.item()*len(yb)
        tot_correct += (logits.argmax(1) == yb).sum().item(); tot_n += len(yb)
    torch.set_grad_enabled(True)
    return tot_loss/tot_n, tot_correct/tot_n

def train_and_track(model, train_loader, val_loader, epochs=20, lr=3e-3, verbose=False):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr); crit = nn.CrossEntropyLoss()
    hist = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}
    for ep in range(epochs):
        tl, ta = _epoch(model, train_loader, crit, opt)
        vl, va = _epoch(model, val_loader,   crit, None)
        hist['loss'].append(tl); hist['accuracy'].append(ta)
        hist['val_loss'].append(vl); hist['val_accuracy'].append(va)
        if verbose: print(f'epoch {ep:2d}  acc {ta:.3f}  val_acc {va:.3f}')
    return hist

def accuracy(model, loader):
    return _epoch(model, loader, nn.CrossEntropyLoss(), None)[1]

## A memoryless baseline (bag-of-embeddings)

Embed every token and **average** over the sequence, then classify. This throws away order — it is the sequence analogue of the bag-of-words model.

In [6]:
class BagOfEmb(nn.Module):
    def __init__(self, vocab=V, d=32, n_class=2):
        super().__init__()
        self.embed = nn.Embedding(vocab, d)
        self.mlp = nn.Sequential(nn.Linear(d, 64), nn.ReLU(), nn.Linear(64, n_class))
    def forward(self, x):
        return self.mlp(self.embed(x).mean(dim=1))     # average over time -> order lost

L = 15
Xtr, ytr = make_recall(4000, L); Xva, yva = make_recall(1000, L)
tl, vl = make_loaders(Xtr, ytr), make_loaders(Xva, yva, shuffle=False)

hist = {}
bag = BagOfEmb(); print('params:', n_params(bag))
hist['BagOfEmb'] = train_and_track(bag, tl, vl, epochs=15)
plotter(hist, metric='accuracy', ylim=[0.0, 1.05])
print('val accuracy:', round(accuracy(bag, vl), 3), ' (chance = 0.5)')

**What is your judgment of this model?** It sits at chance. Both markers are always present, so the *bag* of tokens is identical whichever order they arrived in averaging throws away the one thing that matters. We need a model that processes the sequence **in order** and keeps a memory.

## An RNN, from scratch

A recurrent network keeps a **hidden state** $h^{\langle t \rangle}$ and updates it at each step from the previous state and the current input, exactly the slide equation:

$$h^{\langle t \rangle} = \tanh\!\big(W_{xh}\,x^{\langle t \rangle} + W_{hh}\,h^{\langle t-1 \rangle}\big).$$

We implement the **cell** and then loop over time ourselves, so the recurrence is explicit. For sequence classification we read the **last** hidden state (*many-to-one*).

In [7]:
class RNNCellScratch(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.W_xh = nn.Linear(input_size, hidden_size)              # input  -> hidden
        self.W_hh = nn.Linear(hidden_size, hidden_size, bias=False) # hidden -> hidden
    def forward(self, x_t, h_prev):
        return torch.tanh(self.W_xh(x_t) + self.W_hh(h_prev))       # one time step

class RNNClassifier(nn.Module):
    def __init__(self, vocab=V, d=32, hidden=64, n_class=2):
        super().__init__()
        self.embed  = nn.Embedding(vocab, d)
        self.cell   = RNNCellScratch(d, hidden)
        self.hidden = hidden
        self.head   = nn.Linear(hidden, n_class)
    def forward(self, x):
        emb = self.embed(x)                                        # (N, L, d)
        h = torch.zeros(x.size(0), self.hidden, device=x.device)   # initial state
        for t in range(x.size(1)):                                 # <-- the recurrence
            h = self.cell(emb[:, t], h)
        return self.head(h)                                        # last state -> (N, n_class)

rnn = RNNClassifier(); print('params:', n_params(rnn))
hist['RNN-scratch'] = train_and_track(rnn, tl, vl, epochs=20)
plotter(hist, metric='accuracy', ylim=[0.0, 1.05])
print(f'val accuracy at L={L}:', round(accuracy(rnn, vl), 3))

At `L=15` the from-scratch RNN solves the task — its hidden state carries the order bit from the start to the end. Now the important question: **what happens as the sequence gets longer?**

## The long-range problem: vanishing gradients

To carry the order bit from the start to the final step, the gradient must flow *back* through every intermediate step. Each step multiplies by $W_{hh}$ and a $\tanh$ derivative $\le 1$, so over many steps the signal shrinks toward zero — the **vanishing gradient** problem. The plain RNN simply cannot be trained to remember across long gaps.

We test this directly: train the scratch RNN and the built-in **GRU** and **LSTM** at several sequence lengths and plot accuracy vs. length.

In [8]:
class RNNBuiltin(nn.Module):
    """Sequence classifier around nn.RNN / nn.GRU / nn.LSTM (reads the last output)."""
    def __init__(self, kind, vocab=V, d=32, hidden=64, n_class=2):
        super().__init__()
        self.embed = nn.Embedding(vocab, d)
        self.rnn = {'RNN': nn.RNN, 'GRU': nn.GRU, 'LSTM': nn.LSTM}[kind](d, hidden, batch_first=True)
        if kind == 'LSTM':                       # forget-gate bias -> hold memory
            for name, p in self.rnn.named_parameters():
                if 'bias' in name:
                    H = p.size(0)//4; p.data[H:2*H].fill_(1.0)     # forget gate
        elif kind == 'GRU':                      # update-gate bias -> carry the state (stabilises long-range training)
            for name, p in self.rnn.named_parameters():
                if 'bias' in name:
                    H = p.size(0)//3; p.data[H:2*H].fill_(1.0)     # update gate
        self.head = nn.Linear(hidden, n_class)
    def forward(self, x):
        out, _ = self.rnn(self.embed(x))         # (N, L, hidden)
        return self.head(out[:, -1])             # last time step

lengths = [10, 25, 40]


In [9]:
def train_fresh_and_score(build_fn, Ls, epochs=40):
    Xt, yt = make_recall(3000, Ls); Xv, yv = make_recall(800, Ls)
    ltr, lva = make_loaders(Xt, yt), make_loaders(Xv, yv, shuffle=False)
    model = build_fn()
    train_and_track(model, ltr, lva, epochs=epochs)
    return accuracy(model, lva)

builders = {'RNN-scratch': lambda: RNNClassifier(),
            'GRU':         lambda: RNNBuiltin('GRU'),
            'LSTM':        lambda: RNNBuiltin('LSTM')}
results = {k: [] for k in builders}
for Ls in lengths:
    for name, build in builders.items():
        acc = train_fresh_and_score(build, Ls)
        results[name].append(acc)
        print(f'L={Ls:3d}  {name:12s} val_acc={acc:.2f}')

In [10]:
plt.figure(figsize=(6, 4))
for name, accs in results.items():
    plt.plot(lengths, accs, 'o-', label=name)
plt.axhline(0.5, color='gray', ls=':', label='chance')
plt.xlabel('sequence length'); plt.ylabel('validation accuracy'); plt.ylim(0.3, 1.05)
plt.title('Memory vs. sequence length'); plt.legend(); plt.show()
print('The plain RNN falls to chance as length grows; GRU and LSTM keep remembering.')

## GRU and LSTM — gated memory

The gated cells succeed because they can **choose** what to keep. A **GRU** uses two gates:

$$z = \sigma(W_z[x, h]),\quad r = \sigma(W_r[x, h]),\quad \tilde h = \tanh(W_h[x, r\odot h]),\quad h' = (1-z)\odot h + z\odot \tilde h.$$

The **update gate** $z$ decides how much of the old state to keep vs. replace; the **reset gate** $r$ decides how much past state to use when proposing a new candidate. Because $h'$ can copy $h$ almost unchanged (when $z\approx 0$), information survives many steps. Here is the cell from scratch — this is exactly what `nn.GRU` computes internally.

In [11]:
class GRUCellScratch(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.W_z = nn.Linear(input_size + hidden_size, hidden_size)  # update gate
        self.W_r = nn.Linear(input_size + hidden_size, hidden_size)  # reset gate
        self.W_h = nn.Linear(input_size + hidden_size, hidden_size)  # candidate state
    def forward(self, x_t, h_prev):
        xh = torch.cat([x_t, h_prev], dim=-1)
        z = torch.sigmoid(self.W_z(xh))                             # how much to update
        r = torch.sigmoid(self.W_r(xh))                             # how much past to use
        h_tilde = torch.tanh(self.W_h(torch.cat([x_t, r * h_prev], dim=-1)))
        return (1 - z) * h_prev + z * h_tilde                       # gated blend

print(GRUCellScratch(32, 64))

An **LSTM** goes further: it keeps a separate **cell state** $c$ alongside $h$, with three gates (input, forget, output). The forget gate lets it hold or erase memory over very long ranges. We used the built-in `nn.LSTM` above; the idea is the same — *gates protect memory*.

# Part 2 — Token classification: a label for every step

So far: one label per sequence (*many-to-one*), read from the **last** hidden state. Many tasks — part-of-speech tagging, **named-entity recognition** — need **one label per token** (*many-to-many*). For those we keep **every** hidden state and apply a shared head at each position.

**Task (inside-a-region tagging).** Tokens `0` = `(` (open) and `1` = `)` (close) mark regions; every content token gets label **1 if it lies inside an open region**, else **0**. This is a miniature B-I-O tagging problem: to label a token the model must **remember whether it has seen an unmatched `(`** — pure sequential state.

In [12]:
OPEN, CLOSE = 0, 1        # content tokens are 2..V-1
def make_tagging(n, L, p_open=0.20, p_close=0.25):
    X = np.random.randint(2, V, size=(n, L)).astype(np.int64)
    Y = np.zeros((n, L), dtype=np.int64)
    for i in range(n):
        inside = False
        for t in range(L):
            if not inside and t < L - 2 and np.random.rand() < p_open:
                X[i, t] = OPEN; inside = True; Y[i, t] = 0
            elif inside and np.random.rand() < p_close:
                X[i, t] = CLOSE; inside = False; Y[i, t] = 0
            else:
                Y[i, t] = 1 if inside else 0
    return X, Y

def render(x, y):
    sym = ['(' if t == OPEN else ')' if t == CLOSE else str(t) for t in x]
    print('tokens:', ' '.join(f'{s:>2}' for s in sym))
    print('labels:', ' '.join(f'{v:>2}' for v in y))

Xt, Yt = make_tagging(1, 20); render(Xt[0], Yt[0])

Because the output is now per-token, the training loop compares predictions to labels **at every position** (we flatten the time axis into the loss).

In [13]:
Xtr, Ytr = make_tagging(4000, 25); Xva, Yva = make_tagging(1000, 25)
def tag_loaders(X, Y, bs=128, shuffle=True):
    return DataLoader(TensorDataset(torch.from_numpy(X), torch.from_numpy(Y)), batch_size=bs, shuffle=shuffle)
ttr, tva = tag_loaders(Xtr, Ytr), tag_loaders(Xva, Yva, shuffle=False)

def train_tagger(model, train_loader, val_loader, epochs=12, lr=3e-3):
    model = model.to(device); opt = torch.optim.Adam(model.parameters(), lr); crit = nn.CrossEntropyLoss()
    for ep in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)                                  # (N, L, n_class)
            loss = crit(out.reshape(-1, out.size(-1)), yb.reshape(-1))   # per-token loss
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
    return model

def token_accuracy(model, loader):
    model.eval(); correct = total = 0
    with torch.no_grad():
        for xb, yb in loader:
            pred = model(xb.to(device)).argmax(-1).cpu()
            correct += (pred == yb).sum().item(); total += yb.numel()
    return correct / total

### A memoryless baseline again

A per-token MLP sees only the *current* token, with no memory — so it cannot possibly know whether it is inside a region.

In [14]:
class TokenMLP(nn.Module):
    def __init__(self, vocab=V, d=32, n_class=2):
        super().__init__()
        self.embed = nn.Embedding(vocab, d)
        self.mlp = nn.Sequential(nn.Linear(d, 64), nn.ReLU(), nn.Linear(64, n_class))
    def forward(self, x):
        return self.mlp(self.embed(x))                       # (N, L, n_class), no memory

mlp_tagger = train_tagger(TokenMLP(), ttr, tva)
print('TokenMLP  (no memory)  token accuracy:', round(token_accuracy(mlp_tagger, tva), 3))

### A recurrent tagger
Wrap an RNN and apply the head at **every** step.

In [15]:
class RNNTagger(nn.Module):
    def __init__(self, kind='GRU', vocab=V, d=32, hidden=64, n_class=2):
        super().__init__()
        self.embed = nn.Embedding(vocab, d)
        self.rnn = {'RNN': nn.RNN, 'GRU': nn.GRU, 'LSTM': nn.LSTM}[kind](d, hidden, batch_first=True)
        self.head = nn.Linear(hidden, n_class)
    def forward(self, x):
        out, _ = self.rnn(self.embed(x))                     # (N, L, hidden) -- keep ALL steps
        return self.head(out)                                # (N, L, n_class)

gru_tagger = train_tagger(RNNTagger('GRU'), ttr, tva)
print('GRU tagger            token accuracy:', round(token_accuracy(gru_tagger, tva), 3))

### Did it learn the region logic? Visualise true vs. predicted tags

In [16]:
gru_tagger.eval()
xb = torch.from_numpy(Xva[:4]).to(device)
with torch.no_grad():
    pred = gru_tagger(xb).argmax(-1).cpu().numpy()

fig, axes = plt.subplots(4, 1, figsize=(11, 5))
for i, ax in enumerate(axes):
    strip = np.vstack([Yva[i], pred[i]])                     # row0 = true, row1 = pred
    ax.imshow(strip, aspect='auto', cmap='Greens', vmin=0, vmax=1)
    labels = ['(' if t == OPEN else ')' if t == CLOSE else str(t) for t in Xva[i]]
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, fontsize=8)
    ax.set_yticks([0, 1]); ax.set_yticklabels(['true', 'pred'], fontsize=9)
plt.suptitle('Inside-region tagging: green = inside (label 1)')
plt.tight_layout(); plt.show()
print('The tagger switches on right after "(" and off at ")" -- it is tracking an "inside" bit.')

# Wrap-up

We built the pieces from the slides and watched them work:

* **Embeddings** replace one-hot with dense, learned token vectors.
* An **RNN** carries a hidden state through time; we implemented the recurrence explicitly.
* Plain RNNs suffer **vanishing gradients** and forget across long gaps; **GRU/LSTM** gates protect memory — visible as flat accuracy vs. length while the plain RNN decays to chance.
* **Sequence classification** reads the *last* hidden state (*many-to-one*); **token classification** applies a head at *every* step (*many-to-many*).

**Things to try in class / as exercises**
* Make the RNN **bi-directional** (`bidirectional=True`) — how does token-tagging accuracy change near region edges?
* Stack layers (`num_layers=2`) for a **deep** RNN.
* Push the recall length past 25 — where do GRU and LSTM finally break?
* Allow **nested** brackets in the tagging task and predict *depth* instead of a bit.

**Note.** For real work use the built-in `nn.RNN` / `nn.GRU` / `nn.LSTM` (fast, batched, multi-layer). We wrote the cells by hand only to see the moving parts.